# Gap to the Podium: Phase 4 trajectories

Is India's gap to the podium already there at age 18? Every season best becomes a performance
index (100 = bronze level, higher is better), lined up by the athlete's age. Indian athletes are
compared with the 2024 Olympic finalists (Indian finalists left out).

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.analysis import average_over_games, match_medallists, season_best_bronze
from src.trajectory import pick_thresholds
from src.warehouse import query, table

gap = table("marts.mart_gap_to_podium")
toplists = table("staging.stg_wa_toplists")
podium = query("""
    SELECT championship_year, event_key, athlete_name, country_code, position, mark_value,
           higher_is_better, discipline_group
    FROM staging.stg_olympic_results
    WHERE position BETWEEN 1 AND 3 AND championship_year IN (2016, 2020, 2024)
""")
sb = average_over_games(season_best_bronze(match_medallists(podium, toplists)))
thresholds = pick_thresholds(gap[["event_key", "bronze_threshold"]].merge(sb, on="event_key", how="left"))
print(thresholds["source"].value_counts())
thresholds[thresholds["source"] != "medallists' season bests"]

source
medallists' season bests    42
Name: count, dtype: int64


,event_key,threshold,source


In [2]:
from src.config import RAW_DIR, SEEDS_DIR
from src.trajectory import age_aligned, latest_run, level_by_age, load_group

th = dict(zip(thresholds["event_key"], thresholds["threshold"]))
groups = {"India": ("wa", "india_athletes.csv", None),
          "Finalists": ("wa_finalists", "finalists_2024.csv", "IND")}
traj = {}
for name, (subdir, seed, exclude) in groups.items():
    run = latest_run(RAW_DIR / subdir)
    sbs, births = load_group(run, SEEDS_DIR / seed, exclude_country=exclude)
    traj[name] = age_aligned(sbs, births, th)
    print(f"{name}: run {run.name}, {traj[name]['wa_id'].nunique()} athletes, "
          f"{len(traj[name])} season bests")

India: run 20261006T193207Z, 72 athletes, 672 season bests
Finalists: run 20261011T143113Z, 305 athletes, 3353 season bests


In [3]:
levels = pd.concat({name: level_by_age(t).set_index("age") for name, t in traj.items()}, axis=1)
levels.loc[16:30].round(1)

India                    Finalists                    
    athletes median   q25   q75  athletes median   q25    q75
age                                                          
16        14   85.8  76.2  88.1       112   88.2  80.6   91.3
17        25   86.4  80.8  90.0       155   90.2  84.5   93.4
18        36   88.3  83.5  91.4       190   91.5  85.1   95.5
19        39   88.5  83.1  92.6       206   93.7  89.8   97.0
20        43   89.4  84.9  93.8       238   95.5  91.9   98.4
21        49   92.3  87.5  95.6       244   96.7  92.2   99.3
22        50   92.4  88.2  95.2       250   97.6  94.6   99.8
23        54   93.6  89.9  95.8       246   98.3  96.0  100.1
24        45   94.5  93.0  96.9       235   98.9  96.9  100.4
25        47   95.2  93.7  97.4       226   98.9  97.0  100.6
26        39   93.9  90.8  96.0       200   99.6  97.9  100.8
27        38   96.3  92.8  97.3       186   99.6  97.5  101.3
28        33   94.9  93.3  96.8       157   99.9  97.9  101.2
29        29   94.7  92.2  97.5       147   99.4  97.5  100.9
30        23   95.6  93.1  96.9       117   99.7  97.8  101.2

**First look (median performance index by age).** At 18, Indian athletes are about 3 points
(about 3%) further from bronze level than 2024 finalists (88.3 vs 91.5). The difference grows to
about 5-6 points at 19-20 and then stays at roughly 4-5 points through age 30; finalists level off
near 99-100 (bronze level), Indian athletes near 95. All 42 thresholds are medallists' season
bests. 72 Indian athletes, 305 finalists (Indian finalists left out).

**Caution.** Each age compares whichever athletes have a mark at that age, so the groups change
from age to age. This shows the gap exists at 18 but cannot say whether it grows: Phase 3's H2,
which follows the same athletes season to season, found Indian athletes improve at least as fast.
The mixed-effects model below follows each athlete's own line and settles it.

## 2. Mixed-effects model: does the gap start at 18, and does it grow?

Every athlete-event gets its own level and improvement rate (random effects); each group gets an
average curve with a level at 18, a yearly improvement and a bend (fixed effects). Ages 16-30,
age measured from 18. `india` = gap at 18; `india:years` = difference in yearly improvement.

In [4]:
from src.trajectory import fit_trajectory_model, group_effects, mean_curves, model_frame

frame = model_frame(traj["India"], traj["Finalists"])
result = fit_trajectory_model(frame)
print(f"converged: {result.converged}; {frame['unit'].nunique()} athlete-event lines, "
      f"{len(frame)} season bests")
group_effects(result).round(3)

converged: True; 396 athlete-event lines, 3596 season bests


,term,meaning,estimate,ci_low,ci_high,p_value
0,india,gap at 18 (points),-4.861,-6.749,-2.974,0.000
1,india:years,difference in improvement per year at 18 (points),-0.302,-0.613,0.010,0.058
2,india:I(years ** 2),difference in curvature (points per year squared),0.034,0.012,0.056,0.002


In [5]:
curves = mean_curves(result).pivot(index="age", columns="group", values="perf_index")
curves["gap"] = curves["Finalists"] - curves["India"]
curves.round(1)

group,Finalists,India,gap
age,,,
16,83.8,79.7,4.1
17,86.7,82.1,4.5
18,89.3,84.4,4.9
19,91.6,86.4,5.1
20,93.6,88.2,5.3
21,95.3,89.8,5.5
22,96.8,91.2,5.5
23,97.9,92.4,5.5
24,98.8,93.3,5.5


**Result.** The model (396 athlete-event lines, 3,596 season bests, ages 16-30) puts Indian
athletes 4.9 points behind finalists at age 18 (95% CI 3.0 to 6.7, p < 0.001), larger than the
raw medians (3.2) because it also uses athletes whose age-18 season is missing. Yearly improvement
at 18 is 0.3 points lower (CI -0.61 to +0.01, p = 0.058: not significant), and Indian curves bend
less (+0.034, p = 0.002), so India improves slightly slower until about 22 and slightly faster
after. Over 18-24 this roughly evens out, consistent with Phase 3's H2.

**Conclusion.** The gap is about 4 points at 16, 5 points at 18 and stays at 5-5.5 points through
the development years. It is set before 18 and carried, not built during the development years.
This locates when the gap appears, not why (talent, coaching and competition are not separable
here).

**Caution.** The narrowing after 26 (3.6 points at 30) is not a finding: a quadratic curve is
symmetric by construction, and Indian athletes still competing at 29-30 are likely the stronger
ones.